In [1]:
# AMLC 2025: Multimodal MLP (memory-safe, sparse-first, per-fold densify, AMP, OneCycleLR)

import os, gc, math
import numpy as np
import pandas as pd

from scipy.sparse import load_npz, csr_matrix, hstack
from sklearn.preprocessing import RobustScaler, StandardScaler, normalize
from sklearn.model_selection import KFold
from sklearn.decomposition import TruncatedSVD

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# ---------------- CONFIG ----------------
DATA_PATH = "./data"
MODEL_SAVE_PATH = "./models"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

N_SPLITS = 5
NUM_EPOCHS = 30
BATCH_SIZE = 512
MAX_LR = 2e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0
NUM_WORKERS = 4
SEED = 42

USE_SVD = False
SVD_COMPONENTS = 512
TEXT_EMBED_DIM = 384
TEST_CHUNK_ROWS = 10000   # controls test-time memory

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------------- UTILITIES ----------------
def smape_np(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    num = np.abs(y_pred - y_true)
    mask = denom > 1e-8
    res = np.zeros_like(num, dtype=float)
    res[mask] = num[mask] / denom[mask]
    return 100.0 * np.mean(res)

def assert_finite(name, arr):
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contains non-finite values.")

def to_csr_block(arr2d):
    return csr_matrix(arr2d)

# ---------------- LOAD (sparse-first) ----------------
print("Loading sparse text features (CSR) ...")
X_train_text = load_npz(os.path.join(DATA_PATH, "X_train_text_features_bert.npz")).tocsr()
X_test_text  = load_npz(os.path.join(DATA_PATH,  "X_test_text_features_bert.npz")).tocsr()
print("Text shapes:", X_train_text.shape, X_test_text.shape)

print("Loading ensemble image embeddings ...")
X_train_img = np.load(os.path.join(DATA_PATH, "X_train_img_ensemble.npy"))
X_test_img  = np.load(os.path.join(DATA_PATH, "X_test_img_ensemble.npy"))
print("Image shapes:", X_train_img.shape, X_test_img.shape)

y_train_full = np.load(os.path.join(DATA_PATH, "y_train_full.npy"))
test_ids_df = pd.read_csv(os.path.join(DATA_PATH, "test_ids.csv"))

# ensure 2D
X_train_img = X_train_img.reshape(X_train_img.shape[0], -1)
X_test_img  = X_test_img.reshape(X_test_img.shape[0], -1)
img_cols = X_train_img.shape[1]

# Slice text (sparse) into blocks
if X_train_text.shape[1] <= TEXT_EMBED_DIM:
    raise ValueError("Text feature dimensionality is less than TEXT_EMBED_DIM.")

X_train_text_emb = X_train_text[:, :TEXT_EMBED_DIM]
X_test_text_emb  = X_test_text[:,  :TEXT_EMBED_DIM]
X_train_value    = X_train_text[:, TEXT_EMBED_DIM:TEXT_EMBED_DIM+1]
X_test_value     = X_test_text[:,  TEXT_EMBED_DIM:TEXT_EMBED_DIM+1]
X_train_unit     = X_train_text[:, TEXT_EMBED_DIM+1:]
X_test_unit      = X_test_text[:,  TEXT_EMBED_DIM+1:]

# Convert image arrays and simple stats to sparse CSR for stacking
X_train_img_csr = to_csr_block(X_train_img)
X_test_img_csr  = to_csr_block(X_test_img)
X_train_img_mean = to_csr_block(X_train_img.mean(axis=1, keepdims=True))
X_test_img_mean  = to_csr_block(X_test_img.mean(axis=1, keepdims=True))
X_train_img_std  = to_csr_block(X_train_img.std(axis=1, keepdims=True))
X_test_img_std   = to_csr_block(X_test_img.std(axis=1, keepdims=True))

# Build base design matrices in CSR (memory-light)
X_train_base_sparse = hstack([
    X_train_text_emb, X_train_img_csr, X_train_value,
    X_train_img_mean, X_train_img_std, X_train_unit
], format='csr')

X_test_base_sparse = hstack([
    X_test_text_emb, X_test_img_csr, X_test_value,
    X_test_img_mean, X_test_img_std, X_test_unit
], format='csr')

# Free original big sparse refs
del X_train_text, X_test_text
gc.collect()

print("Base CSR shapes:", X_train_base_sparse.shape, X_test_base_sparse.shape)

# ---------------- MODEL ----------------
class MultimodalPricePredictor(nn.Module):
    def __init__(self, input_size, text_size=384, img_size=512):
        super().__init__()
        self.text_size = text_size
        self.img_size = img_size
        other_size = input_size - text_size - img_size
        if other_size <= 0:
            raise ValueError("Computed other_size <= 0. Check sizes.")

        self.text_branch = nn.Sequential(
            nn.Linear(text_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
        )

        self.img_branch = nn.Sequential(
            nn.Linear(img_size, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(384, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )

        self.other_branch = nn.Sequential(
            nn.Linear(other_size, 64),
            nn.LayerNorm(64),
            nn.GELU(),
        )

        fusion_size = 128 + 256 + 64
        self.fusion = nn.Sequential(
            nn.Linear(fusion_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.output = nn.Linear(64, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                if hasattr(m, "weight") and m.weight is not None:
                    nn.init.constant_(m.weight, 1.0)
                if hasattr(m, "bias") and m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        t = self.text_branch(x[:, :self.text_size])
        i = self.img_branch(x[:, self.text_size:self.text_size + self.img_size])
        o = self.other_branch(x[:, self.text_size + self.img_size:])
        fused = self.fusion(torch.cat([t, i, o], dim=1))
        return self.output(fused)

# ---------------- TRAIN / CV ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = device == "cuda"
print("Device:", device)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_preds_log = np.zeros(X_train_base_sparse.shape[0], dtype=np.float32)
all_fold_test_preds = []
all_scores = []

# column ranges for image block inside the CSR matrices
img_slice_start = TEXT_EMBED_DIM
img_slice_end = TEXT_EMBED_DIM + img_cols

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_base_sparse, y_train_full), start=1):
    print(f"\n===== FOLD {fold}/{N_SPLITS} =====")

    # Subset rows as CSR; densify only per fold
    X_tr_raw = X_train_base_sparse[tr_idx]
    X_va_raw = X_train_base_sparse[va_idx]

    # Optional SVD on image columns
    if USE_SVD:
        svd = TruncatedSVD(n_components=min(SVD_COMPONENTS, img_cols), random_state=SEED)
        svd.fit(X_tr_raw[:, img_slice_start:img_slice_end])
        X_tr_img_reduced = svd.transform(X_tr_raw[:, img_slice_start:img_slice_end])
        X_va_img_reduced = svd.transform(X_va_raw[:, img_slice_start:img_slice_end])
        X_test_img_reduced = svd.transform(X_test_base_sparse[:, img_slice_start:img_slice_end])
        img_reduced_dim = X_tr_img_reduced.shape[1]
    else:
        # Densify per fold
        X_tr_img_reduced = X_tr_raw[:, img_slice_start:img_slice_end].toarray()
        X_va_img_reduced = X_va_raw[:, img_slice_start:img_slice_end].toarray()
        img_reduced_dim = img_cols
        # test handled in chunks later

    # Densify remaining columns for current fold
    X_tr_left = hstack([
        X_tr_raw[:, :TEXT_EMBED_DIM],
        X_tr_raw[:, img_slice_end:]
    ], format='csr').toarray()

    X_va_left = hstack([
        X_va_raw[:, :TEXT_EMBED_DIM],
        X_va_raw[:, img_slice_end:]
    ], format='csr').toarray()

    # Recompose dense per-fold arrays: [text_emb, img_block, rest]
    X_tr_fold = np.hstack([X_tr_left[:, :TEXT_EMBED_DIM], X_tr_img_reduced, X_tr_left[:, TEXT_EMBED_DIM:]])
    X_va_fold = np.hstack([X_va_left[:, :TEXT_EMBED_DIM], X_va_img_reduced, X_va_left[:, TEXT_EMBED_DIM:]])

    # Per-sample L2 normalization of embeddings (text + image)
    def l2_norm_embeds(A, text_dim, img_dim):
        emb = A[:, :text_dim + img_dim]
        emb = normalize(emb, norm="l2", axis=1)
        A[:, :text_dim + img_dim] = emb
        return A

    X_tr_fold = l2_norm_embeds(X_tr_fold, TEXT_EMBED_DIM, img_reduced_dim)
    X_va_fold = l2_norm_embeds(X_va_fold, TEXT_EMBED_DIM, img_reduced_dim)

    # Per-fold scaling
    scaler_rob = RobustScaler()
    X_tr_scaled = scaler_rob.fit_transform(X_tr_fold)
    X_va_scaled = scaler_rob.transform(X_va_fold)

    sc2 = StandardScaler()
    X_tr_scaled = sc2.fit_transform(X_tr_scaled)
    X_va_scaled = sc2.transform(X_va_scaled)

    assert_finite("X_tr_scaled", X_tr_scaled)
    assert_finite("X_va_scaled", X_va_scaled)

    y_tr_log = np.log1p(y_train_full[tr_idx].astype(np.float32))
    y_va_log = np.log1p(y_train_full[va_idx].astype(np.float32))

    train_ds = TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32),
                             torch.tensor(y_tr_log, dtype=torch.float32).view(-1, 1))
    val_ds = TensorDataset(torch.tensor(X_va_scaled, dtype=torch.float32),
                           torch.tensor(y_va_log, dtype=torch.float32).view(-1, 1))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS>0))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS>0))

    input_size = X_tr_scaled.shape[1]
    model = MultimodalPricePredictor(input_size=input_size,
                                     text_size=TEXT_EMBED_DIM,
                                     img_size=img_reduced_dim).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = max(1, len(train_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, steps_per_epoch=steps_per_epoch, epochs=NUM_EPOCHS, anneal_strategy="cos"
    )

    criterion = nn.HuberLoss(delta=1.0)
    scaler_amp = torch.cuda.amp.GradScaler() if device == "cuda" else None

    best_val_smape = float("inf")
    patience, patience_counter = 7, 0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        tr_loss = 0.0
        for xb, yb in tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch}/{NUM_EPOCHS} [Train]", leave=False):
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if scaler_amp is not None:
                with torch.cuda.amp.autocast():
                    out = model(xb); loss = criterion(out, yb)
                scaler_amp.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler_amp.step(optimizer); scaler_amp.update()
            else:
                out = model(xb); loss = criterion(out, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            tr_loss += loss.item()
            scheduler.step()
        tr_loss /= max(1, len(train_loader))

        model.eval()
        va_loss = 0.0; preds=[]; targets=[]
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
                out = model(xb); loss = criterion(out, yb)
                va_loss += loss.item()
                preds.append(out.cpu().numpy()); targets.append(yb.cpu().numpy())
        va_loss /= max(1, len(val_loader))
        preds = np.vstack(preds).flatten(); targets = np.vstack(targets).flatten()

        val_pred_price = np.expm1(preds); val_true_price = np.expm1(targets)
        val_pred_price[~np.isfinite(val_pred_price)] = 0
        val_pred_price = np.maximum(val_pred_price, 0)
        val_smape = smape_np(val_true_price, val_pred_price)

        print(f"Epoch {epoch:02d} | Train {tr_loss:.4f} | Val {va_loss:.4f} | SMAPE {val_smape:.4f}%")

        if val_smape < best_val_smape:
            best_val_smape = val_smape; patience_counter = 0
            torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, f"model_fold_{fold}_best.pth"))
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # Load best for OOF and test
    model.load_state_dict(torch.load(os.path.join(MODEL_SAVE_PATH, f"model_fold_{fold}_best.pth"),
                                     map_location=device))
    model.eval()

    # OOF preds
    oof_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    oof_list = []
    with torch.no_grad():
        for xb, _ in oof_loader:
            xb = xb.to(device, non_blocking=True)
            out = model(xb).cpu().numpy()
            oof_list.append(out)
    preds_log = np.vstack(oof_list).flatten().astype(np.float32)
    oof_preds_log[va_idx] = preds_log

    va_pred_prices = np.expm1(preds_log); va_pred_prices[~np.isfinite(va_pred_prices)] = 0
    va_pred_prices = np.maximum(va_pred_prices, 0)
    va_true_prices = np.expm1(y_va_log)
    fold_score = smape_np(va_true_prices, va_pred_prices)
    all_scores.append(fold_score)
    print(f"Fold {fold} Final SMAPE: {fold_score:.4f}% (best {best_val_smape:.4f}%)")

    # Test predictions in chunks to cap memory
    fold_test_preds = []
    with torch.no_grad():
        for start in range(0, X_test_base_sparse.shape[0], TEST_CHUNK_ROWS):
            end = min(start + TEST_CHUNK_ROWS, X_test_base_sparse.shape[0])
            X_chunk_csr = X_test_base_sparse[start:end]

            if USE_SVD:
                img_block = svd.transform(X_chunk_csr[:, img_slice_start:img_slice_end])
                left_block = hstack([X_chunk_csr[:, :TEXT_EMBED_DIM],
                                     X_chunk_csr[:, img_slice_end:]], format='csr').toarray()
            else:
                img_block = X_chunk_csr[:, img_slice_start:img_slice_end].toarray()
                left_block = hstack([X_chunk_csr[:, :TEXT_EMBED_DIM],
                                     X_chunk_csr[:, img_slice_end:]], format='csr').toarray()

            X_chunk = np.hstack([left_block[:, :TEXT_EMBED_DIM], img_block, left_block[:, TEXT_EMBED_DIM:]])
            X_chunk[:, :TEXT_EMBED_DIM + img_reduced_dim] = normalize(
                X_chunk[:, :TEXT_EMBED_DIM + img_reduced_dim], norm="l2", axis=1
            )

            X_chunk = scaler_rob.transform(X_chunk)
            X_chunk = sc2.transform(X_chunk)

            ds = TensorDataset(torch.tensor(X_chunk, dtype=torch.float32))
            dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            chunk_preds=[]
            for (xb,) in dl:
                xb = xb.to(device, non_blocking=True)
                out = model(xb).cpu().numpy()
                chunk_preds.append(out)
            fold_test_preds.append(np.vstack(chunk_preds))

            del X_chunk, left_block, img_block, ds, dl
            gc.collect()
            if device == "cuda": torch.cuda.empty_cache()

    fold_test_preds = np.vstack(fold_test_preds).flatten().astype(np.float32)
    all_fold_test_preds.append(fold_test_preds)

    # cleanup per-fold
    del model, optimizer, scheduler, scaler_rob, sc2
    del X_tr_raw, X_va_raw, X_tr_fold, X_va_fold, X_tr_left, X_va_left
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

# ---------------- CV REPORT ----------------
print("\n===== CROSS-VALIDATION RESULTS =====")
print(f"Average SMAPE: {np.mean(all_scores):.4f}%")
print(f"Std Dev:      {np.std(all_scores):.4f}%")
print(f"Min SMAPE:    {np.min(all_scores):.4f}%")
print(f"Max SMAPE:    {np.max(all_scores):.4f}%")

# ---------------- ENSEMBLE TEST PREDICTIONS ----------------
all_fold_test_preds = np.vstack(all_fold_test_preds)
ensembled_log = np.mean(all_fold_test_preds, axis=0).astype(np.float32)
final_predictions = np.expm1(ensembled_log)
final_predictions[~np.isfinite(final_predictions)] = 0
final_predictions = np.maximum(final_predictions, 0)

submission_df = pd.DataFrame({
    "sample_id": test_ids_df["sample_id"],
    "price": final_predictions
})
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"\nSaved: {submission_path}")
print(f"Pred range: [{final_predictions.min():.2f}, {final_predictions.max():.2f}]")
print(f"Pred mean: {final_predictions.mean():.2f}, median: {np.median(final_predictions):.2f}")


Loading sparse text features (CSR) ...
Text shapes: (75000, 483) (75000, 483)
Loading ensemble image embeddings ...
Image shapes: (75000, 512) (75000, 512)
Base CSR shapes: (75000, 997) (75000, 997)
Device: cuda

===== FOLD 1/5 =====


C:\Users\Deepak\AppData\Local\Temp\ipykernel_17808\1565265256.py:286: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler() if device == "cuda" else None
Fold 1 Epoch 1/30 [Train]:   0%|                                                                                                   | 0/118 [00:00<?, ?it/s]C:\Users\Deepak\AppData\Local\Temp\ipykernel_17808\1565265256.py:298: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\Deepak\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of 

Epoch 01 | Train 0.6418 | Val 0.3568 | SMAPE 69.5044%


Epoch 02 | Train 0.3504 | Val 0.3039 | SMAPE 63.6181%


Epoch 03 | Train 0.3023 | Val 0.2800 | SMAPE 60.4365%


...

Epoch 21 | Train 0.0968 | Val 0.2215 | SMAPE 50.9299%


Epoch 22 | Train 0.0927 | Val 0.2212 | SMAPE 50.8219%


Epoch 23 | Train 0.0901 | Val 0.2238 | SMAPE 50.9849%


Epoch 24 | Train 0.0870 | Val 0.2216 | SMAPE 50.7860%


Epoch 25 | Train 0.0849 | Val 0.2221 | SMAPE 50.7856%
Early stopping at epoch 25
Fold 5 Final SMAPE: 50.7651% (best 50.7651%)

===== CROSS-VALIDATION RESULTS =====
Average SMAPE: 50.8481%
Std Dev:      0.4727%
Min SMAPE:    50.0372%
Max SMAPE:    51.4752%

Saved: submission.csv
Pred range: [1.21, 192.58]
Pred mean: 19.61, median: 13.53
